##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [1]:
import cv2
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Dense,
    LSTM,
    TimeDistributed,
    Dropout,
    BatchNormalization,
    GlobalAveragePooling2D,
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [3]:
target = r"lab07-action-recognition\action"

classes = ["biking", "horse_riding", "tennis_swing"]

os.makedirs(target, exist_ok=True)

In [ ]:
for cls in classes:
    print(cls, len(os.listdir(os.path.join(target, cls))))

SyntaxError: unexpected character after line continuation character (2571551805.py, line 2)

In [ ]:
def extract_frames(video_path, output_folder):
    cap = cv2.VideoCapture(video_path)
    count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame = cv2.resize(frame, (224, 224))
        cv2.imwrite(f"{output_folder}/frame_{count}.jpg", frame)
        count += 1
    
    cap.release()

In [ ]:
train_paths, test_paths = train_test_split(data_paths, test_size=0.2, random_state=42)

In [ ]:
model = Sequential([
    # Keep the CNN small and pool spatial features instead of flattening huge maps.
    TimeDistributed(
        Conv2D(16, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(1e-4)),
        input_shape=(30, 224, 224, 3)
    ),
    TimeDistributed(BatchNormalization()),
    TimeDistributed(MaxPooling2D(2, 2)),
    TimeDistributed(Dropout(0.25)),

    TimeDistributed(Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=l2(1e-4))),
    TimeDistributed(BatchNormalization()),
    TimeDistributed(MaxPooling2D(2, 2)),
    TimeDistributed(Dropout(0.25)),

    TimeDistributed(GlobalAveragePooling2D()),

    LSTM(32, dropout=0.3, recurrent_dropout=0.2, kernel_regularizer=l2(1e-4)),
    Dense(32, activation='relu', kernel_regularizer=l2(1e-4)),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

In [ ]:
# Make a validation set from the training data so the test set stays unbiased.
stratify_labels = train_labels.argmax(axis=1) if len(train_labels.shape) > 1 else train_labels
train_data, val_data, train_labels, val_labels = train_test_split(
    train_data,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=stratify_labels
)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

history = model.fit(
    train_data,
    train_labels,
    epochs=50,
    batch_size=4,
    validation_data=(val_data, val_labels),
    callbacks=callbacks
)

In [ ]:
student_name = "Raneem Alqahtani" 
save_path = "Raneem_Alqahtani_ucf11_model.h5"
model.save(save_path)
print(f"Model saved as {save_path}")

In [ ]:
# validate the model on a new video
video_frames = extract_frames("new_video.mp4", "temp_frames")
prediction = model.predict(video_frames)
print("Predicted action:", prediction)